# DECISIONS ANTERIORS

## SEPARACIÓ NUMERIQUES/ CATEGORIQUES

In [ ]:
# Variables numèriques (continuen i discretes)
num_features = [
    'age', 'bmi', 'systolic_bp', 'diastolic_bp', 'heart_rate', 
    'respiratory_rate', 'temperature_c', 'spo2', 'glucose_mg_dl', 
    'creatinine_mg_dl', 'sodium_mmol_l', 'hemoglobin_g_dl', 
    'num_medications', 'prior_visits_12m', 'length_of_stay_days'
]

# Variables categòriques (incloses les binàries de tipus text)
cat_features = [
    'sex', 'smoking_status', 'encounter_type', 'diabetes', 
    'hypertension', 'chronic_kidney_disease', 'primary_diagnosis', 
    'discharge_destination', 'readmitted_30d', 'label'
]

# Descartem 'id'

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer

# 3. Preprocessing de variables numéricas
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),  # variables buides-> les emplenem posant 'median' a tot arreu
    ("scaler", RobustScaler())                    # standarització ( hi han molts outliers en algunes, en el futur ho dividim segons això)
])

# 4. Preprocessing de variables categóricas
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="constant",
        fill_value="Missing"
    )),
    #("imputer", SimpleImputer(strategy="most_frequent")),   # variables buides-> les emplenem posant 'most frequent' a tot arreu
    ("encoder", OneHotEncoder(handle_unknown="ignore"))     # ara tot onehot encoder per la primera prova, despres separem 
])

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])


## VARIABLE AGE

age group separation 1:

```python
train["age_group"] = pd.cut(
    train["age"],
    bins=[-float("inf"), 30, 40, 50, 60, 70, 80, float("inf")],
    labels=["<30", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"],
    right=False
)
```

age group separation 2:

```python
train["age_group"] = pd.cut(
    train["age"],
    bins=[-float("inf"), 40, 60, 70, 80, float("inf")],
    labels=["<40", "40-59", "60-69", "70-79", "80+"],
    right=False
)
```

age group separation 3:

```python
bins = [0, 40, 60, 80, float("inf")]

train["age_group"] = pd.cut(
    train["age"],
    bins=bins,
    labels=False,
    right=False
)

# Calcular las medias usando solo los datos de entrenamiento
age_means = train.groupby("age_group", observed=True)["age"].mean()

# Sustituir cada grupo por su media
train["age_group_mean"] = train["age_group"].map(age_means)
```

CAP HA MILLORAT EL RESULTAT

## GRID SEARCH EVAL

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {
    "classifier__C": [0.1, 1, 10, 100],
}

grid = GridSearchCV(
    model,
    param_grid=param_grid,
    scoring="accuracy",  # Usa la métrica oficial del reto
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best C:", grid.best_params_)
print("Best CV score:", grid.best_score_)